In [1]:
import sys
sys.executable

'/usr/local/opt/python@3.11/bin/python3.11'

In [87]:
import ast
import pandas as pd
import numpy as np
from pygit2 import Object, Repository, GIT_SORT_TIME
from pygit2 import init_repository, Patch
from colorama import Fore
from tqdm import tqdm
import swifter
from pandarallel import pandarallel
from bs4 import BeautifulSoup, SoupStrainer
import requests
from urllib.request import urlopen
import re


In [3]:
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [4]:
library = 'lablup/backend.ai'

In [19]:
# import all push data
df_pr = pd.DataFrame()
commit_urls = []
for val in np.arange(0, 100, 1):
    if val < 10:
        val = "0" + str(val)
    try:
        df_part = pd.read_csv(f'data/github_clean/prEvent0000000000{val}.csv', index_col = 0)
        df_part['partition'] = val
        df_pr = pd.concat([df_pr, df_part])
    except:
        print(f'data/github_clean/prEvent0000000000{val}.csv not found')

In [102]:
def grabCommits(repo, pr_commits_url):
    try:
        pull_info = "/".join(pr_commits_url.split("/")[-3:-1]).replace("pulls","pull")
        scrape_url = f"https://github.com/{repo}/{pull_info}/commits"

        product = SoupStrainer('div', {'id': 'commits_bucket'})
    
        page = urlopen(scrape_url)
        soup = BeautifulSoup(page.read(),parse_only = product)
        commits = soup.find_all("a", attrs={"id":re.compile(r'commit-details*')})
        commit_urls = [c['href'].split("/")[-1] for c in commits]
        return commit_urls
    except Exception as e:
        return e.args[0]

def grabCommitsBetter(repo, pr_commits_url):
    res = grabCommits(repo, pr_commits_url)
    while type(res) != list:
        time.sleep(1)
        res = grabCommits(repo, pr_commits_url)

In [125]:
t = pd.read_csv('data/github_clean/prEventCommits000000000000.csv',index_col = 0)

In [134]:
t[t['commit_list'].apply(lambda x: x[0] == '[')]

,type,created_at,repo_id,repo_name,actor_id,actor_login,org_id,org_login,pr_id,pr_node_id,pr_title,pr_state,pr_locked,pr_number,pr_body,pr_issue_url,pr_merged_at,pr_closed_at,pr_updated_at,pr_commits,pr_additions,pr_deletions,pr_changed_files,pr_author_association,pr_assignees,pr_requested_reviewers,pr_requested_teams,pr_merged_by_login,pr_merged_by_id,pr_merged_by_type,pr_merged_by_site_admin,pr_label,pr_patch_url,pr_commits_url,partition,commit_list
1,PullRequestEvent,2023-06-18 16:44:04+00:00,104200746,python-openapi/openapi-core,1679024,p1c2u,126442889.0,python-openapi,1397201387,PR_kwDOBjX6Ks5TR5nr,Bump codecov/codecov-action from 1 to 3,closed,False,601,Bumps [codecov/codecov-action](https://github....,https://api.github.com/repos/python-openapi/op...,2023-06-18T16:44:03Z,2023-06-18T16:44:04Z,2023-06-18T16:44:04Z,1,1,1,1,CONTRIBUTOR,[],[],[],p1c2u,1679024.0,User,False,"['dependencies', 'github_actions']",https://github.com/python-openapi/openapi-core...,https://api.github.com/repos/python-openapi/op...,0,['60b0f71036b3ca8fbbffc4ad34d920dbb6e3fba5']
14,PullRequestEvent,2022-11-01 01:52:07+00:00,4127088,Azure/azure-sdk-for-python,7108081,banibrata,6844498.0,Azure,1105894802,PR_kwDOAD75cM5B6p2S,[ML] Environment isinstance() check on __eq__,closed,False,27201,# Description\r\n\r\nPlease add an informative...,https://api.github.com/repos/Azure/azure-sdk-f...,2022-11-01T01:52:06Z,2022-11-01T01:52:06Z,2022-11-01T01:52:06Z,4,16,14,1,MEMBER,[],"[{'login': 'hugoaponte', 'id': 13742976, 'node...",[],banibrata,7108081.0,User,False,[],https://github.com/Azure/azure-sdk-for-python/...,https://api.github.com/repos/Azure/azure-sdk-f...,0,"['321dc1345413c8ab50a9e239306885a533ccc29b', '..."
17,PullRequestEvent,2023-05-03 00:10:12+00:00,44974847,fabiocaccamo/django-admin-interface,41898282,github-actions[bot],NaN,NaN,1335965685,PR_kwDOAq5C_85PoTf1,Update pre-commit hooks,open,False,277,Update versions of pre-commit hooks to latest ...,https://api.github.com/repos/fabiocaccamo/djan...,NaN,NaN,2023-05-03T00:10:11Z,1,1,1,1,CONTRIBUTOR,[],[],[],NaN,NaN,NaN,NaN,[],https://github.com/fabiocaccamo/django-admin-i...,https://api.github.com/repos/fabiocaccamo/djan...,0,['1950f3ed93c70550b2bc6511fecbf491c2bcc47a']
18,PullRequestEvent,2023-05-03 22:05:16+00:00,67641996,oneapi-src/oneTBB,59893122,dnmokhov,60144784.0,oneapi-src,1337456238,PR_kwDOBAgijM5Pt_Zu,Add DLL info for all tbbbind versions instead ...,open,False,1096,### Description \r\n_Add a comprehensive descr...,https://api.github.com/repos/oneapi-src/oneTBB...,NaN,NaN,2023-05-03T22:05:15Z,1,1,3,1,CONTRIBUTOR,[],[],[],NaN,NaN,NaN,NaN,[],https://github.com/oneapi-src/oneTBB/pull/1096...,https://api.github.com/repos/oneapi-src/oneTBB...,0,"['a161276771649273fc413ed0448c67bf35492107', '..."
20,PullRequestEvent,2021-08-12 16:38:52+00:00,91379993,tensorflow/tensorboard,2322480,psybuzz,15658638.0,tensorflow,711229371,MDExOlB1bGxSZXF1ZXN0NzExMjI5Mzcx,timeseries: restore line charts that lose webg...,open,False,5237,"Googlers, see b/196294346.",https://api.github.com/repos/tensorflow/tensor...,NaN,NaN,2021-08-12T16:38:52Z,1,89,3,12,COLLABORATOR,[],[],[],NaN,NaN,NaN,NaN,[],https://github.com/tensorflow/tensorboard/pull...,https://api.github.com/repos/tensorflow/tensor...,0,['c2ececa31d1fae27277e760cea56421d0edd4801']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115555,PullRequestEvent,2022-02-08 19:55:33+00:00,112776537,scoutapp/scout_apm_python,1281215,tim-schilling,458509.0,scoutapp,842951184,PR_kwDOBrjVWc4yPmoQ,Version 2.24.0,closed,False,721,NaN,https://api.github.com/repos/scoutapp/scout_ap...,2022-02-08T19:55:33Z,2022-02-08T19:55:33Z,2022-02-08T19:55:33Z,1,5,1,2,COLLABORATOR,[],[],[],tim-schilling,1281215.0,User,False,[],https://github.com/scoutapp/scout_apm_python/p...,https://api.github.com/repos/scoutapp/scout_ap...,0,['7fc43be42bb9484998c93c4e69965e701b47c886']
115692,PullRequestEvent,2021-1

In [133]:
t.loc[[118416]].apply(lambda x: grabCommits(x['repo_name'], x['pr_commits_url']), axis = 1)

118416    [13202df54a8a527c5faea69516fc8fb99ba602b8, 1b0...
dtype: object

In [386]:
df_library = df_push[df_push['repo_name'] == library]
df_library.loc[:,'commit_urls'] = df_library['commit_urls'].apply(lambda x: ast.literal_eval(x))
commit_urls = df_library['commit_urls'].explode().tolist()

In [387]:
repo = Repository('../repos/backend.ai')
sum = 0
for commit in commit_urls:
    if type(commit) != float:
        sha = commit.split("/")[-1]
        commit_a = repo.get(sha)
        if type(commit_a) != type(None):
            sum += 1
print(f"{sum} out of {len(commit_urls)} commit SHAs can be found ({100*round(sum/len(commit_urls), 3)}%)")

21523 out of 25698 commit SHAs can be found (83.8%)


In [388]:
def createCommitGroup(push_before, push_head, commit_urls, push_size):
    if push_size == 0:
        return [[]]
    elif push_size == 1:
        return [[push_before, push_head]]
    else:
        avail_commits = len(commit_urls)
        lst_result = [[push_before, commit_urls[0].split("/")[-1]]]
        for i in range(avail_commits-1):
            lst_result.append([commit_urls[i].split("/")[-1], commit_urls[i+1].split("/")[-1]])
        return lst_result

In [389]:
df_library['commit_groups'] = \
    df_library.apply(lambda x: createCommitGroup(x['push_before'], x['push_head'], x['commit_urls'], x['push_size']), axis = 1)
df_commit_groups = df_library[['push_id', 'repo_id', 'repo_name', 'actor_id', 'actor_login', 
                               'org_id', 'org_login', 'push_size', 'push_before', 'push_head',
                               'commit_groups']].explode('commit_groups')

/var/folders/5s/dvrxt95949x1pm_sjxm85lj00000gn/T/ipykernel_39288/1534266405.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_library['commit_groups'] = \


In [390]:
def returnCommitStats(x):
    if len(x) < 2:
        return []
    commit_parent_sha = x[0]
    commit_head_sha = x[1]
    commit_parent = repo.get(commit_parent_sha)
    commit_head = repo.get(commit_head_sha)
    if type(commit_parent) != type(None) and type(commit_head) != type(None):
        diff = repo.diff(commit_parent, commit_head, context_lines=0, interhunk_lines=0)
        commit_sha = commit_head_sha
        commit_author_name = commit_head.author.name
        commit_author_email = commit_head.author.email
        committer_author_name = commit_head.committer.name
        committer_author_email = commit_head.committer.email
        commit_message = commit_head.message
        commit_additions = diff.stats.insertions
        commit_deletions = diff.stats.deletions
        commit_changes_total = commit_additions + commit_deletions
        commit_files_changed_count = diff.stats.files_changed
        commit_time = commit_head.commit_time
        commit_file_changes = []
        
        for obj in diff:
            if type(obj) == Patch:
                additions = 0
                deletions = 0
                for hunk in obj.hunks:
                  for line in hunk.lines:
                    # The new_lineno represents the new location of the line after the patch. If it's -1, the line has been deleted.
                    if line.new_lineno == -1: 
                        deletions += 1
                    # Similarly, if a line did not previously have a place in the file, it's been added fresh. 
                    if line.old_lineno == -1: 
                        additions += 1
                commit_file_changes.append({'file':obj.delta.new_file.path,
                                            'additions': additions,
                                            'deletions': deletions,
                                            'total': additions + deletions})
        return [commit_sha, commit_author_name, commit_author_email, committer_author_name, committer_author_email,
                commit_message, commit_additions, commit_deletions, commit_changes_total, commit_files_changed_count,
                commit_file_changes]
    return []

In [392]:
%%time
commit_data = df_commit_groups['commit_groups'].parallel_apply(returnCommitStats)

CPU times: user 1.54 s, sys: 1.02 s, total: 2.56 s
Wall time: 12min 53s


In [393]:
df_commit = pd.DataFrame(commit_data.tolist(),
                        columns = ['commit sha', 'commit author name', 'commit author email', 'committer name',
                                   'commmitter email', 'commit message', 'commit additions', 'commit deletions',
                                   'commit changes total', 'commit files changed count', 'commit file changes'])

In [394]:
df_commit_final = pd.concat([df_commit_groups.reset_index(drop = True), df_commit], axis = 1)
df_commit_final

,push_id,repo_id,repo_name,actor_id,actor_login,org_id,org_login,push_size,push_before,push_head,...,commit author name,commit author email,committer name,commmitter email,commit message,commit additions,commit deletions,commit changes total,commit files changed count,commit file changes
0,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
1,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Joongi Kim,joongi@lablup.com,GitHub,noreply@github.com,setup: Remove dependency to psycopg (#702)\n\n...,204.0,296.0,500.0,21.0,"[{'file': 'changes/702.breaking.md', 'addition..."
2,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Sanghun Lee,oper6909@gmail.com,GitHub,noreply@github.com,feat: Do not return AgentSummary if `hide-agen...,10.0,2.0,12.0,2.0,"[{'file': 'changes/699.feature.md', 'additions..."
3,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Jonghyun Park,jpark@lablup.com,GitHub,noreply@github.com,feat: add `hide_agents` webserver config (#704...,6.0,1.0,7.0,4.0,"[{'file': 'changes/704.feature', 'additions': ..."
4,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,DaeHyun Sung,dhsung@lablup.com,GitHub,noreply@github.com,docs: update python default version (#724)\n\n...,1448.0,996.0,2444.0,11.0,"[{'file': 'docs/README.md', 'additions': 1, 'd..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25693,11646000079,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,286,6f1c5a2c203da153978b9b334709b33d6ba0383a,a2e85fd58c9d7fe10404bc26f6966f20c9097040,...,Jeongseok Kang,piono623@naver.com,GitHub,noreply@github.com,fix: Update integration test to work with the ...,96.0,55.0,151.0,8.0,"[{'file': 'changes/442.fix.md', 'additions': 1..."
25694,13838497595,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,1,437e4625f172057e645465c7084492cfeb0143e8,8e26fd42b6ec476b084f237b3b7a4a5c2f8e0c8f,...,Kyujin Cho,kyujin.cho@lablup.com,GitHub,noreply@github.com,feat: update POST `/service` API to also accep...,85.0,38.0,123.0,3.0,[{'file': 'src/ai/backend/client/cli/service.p...
25695,10610526310,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,1,37c8df3174c768b96f542e6511f33ffad219079e,b1c7115142f78f43743f85f417f1f7c852116dc5,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
25696,10753912953,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,1,6eb9396d72c19b8388a1dec5f9749192f59c1f97,cc5fc1d3b750ad1cae16d18d01544efcc2453caf,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None


In [418]:
df_repos

,github repo,count
0,dagster-io/dagster,43538
1,Azure/azure-sdk-for-python,30160
2,DataDog/dd-trace-py,26048
3,pulumi/pulumi,21919
4,ray-project/ray,20780
...,...,...
5846,hellock/cvbase,1
5847,heroku/heroku.py,1
5848,DenisCarriere/geocoder,1
5849,nickoala/telepot,1


In [417]:
df_repos = pd.DataFrame(df_push[['repo_name']].value_counts()).reset_index().rename(
    {'repo_name': 'github repo'}, axis = 1)
df_repos.to_csv('data/inputs/github_repo_grab_commits.csv')

In [430]:
import subprocess
lib = 'dagster-io/dagster.git'
lib_p2 = lib.split("/")[1]
lib_ren = lib.replace("/","___")
subprocess.Popen(["git", "clone", f"https://github.com/{lib}", f"{lib_ren}"], cwd = "../repos").wait()

Cloning into 'dagster-io___dagster.git'...
Updating files: 100% (7050/7050), done.


0